# LaflaAi-Core 100 Step Checkpoint Test

Bu defter yerel egitim yapmaz. Colab GPU uzerinde zip'i acar, egitimi 100 step ile sinirlar, `lafla-step-000100` checkpoint'ini test eder ve sonucu Drive'a arsivler.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
from pathlib import Path
import shutil
import zipfile

zip_candidates = [
    Path('/content/LaflaAi-Core.zip'),
    Path('/content/LaflaAi-Core-step100-ready.zip'),
    Path('/content/gdrive/MyDrive/LaflaAi-Core.zip'),
    Path('/content/gdrive/MyDrive/LaflaAi-Core-step100-ready.zip'),
    Path('/content/gdrive/MyDrive/LaflaAI/LaflaAi-Core.zip'),
    Path('/content/gdrive/MyDrive/LaflaAI/LaflaAi-Core-step100-ready.zip'),
]
zip_path = next((path for path in zip_candidates if path.exists()), None)
if zip_path is None:
    raise FileNotFoundError('LaflaAi-Core zip bulunamadi. Zip dosyasini /content veya MyDrive/LaflaAI altina yukleyin.')

repo = Path('/content/LaflaAi-Core')
if repo.exists():
    shutil.rmtree(repo)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall('/content')
assert repo.exists(), 'Zip icinde LaflaAi-Core klasoru olmali.'
print('Acilan zip:', zip_path)
%cd /content/LaflaAi-Core

In [ ]:
!python -m pip install -r requirements/colab.txt
!nvidia-smi

In [ ]:
from pathlib import Path

Path('/content/LaflaAI/tokenizer').mkdir(parents=True, exist_ok=True)
Path('/content/LaflaAI/reports').mkdir(parents=True, exist_ok=True)
Path('/content/LaflaAI/checkpoints').mkdir(parents=True, exist_ok=True)
Path('/content/LaflaAI/hf-package').mkdir(parents=True, exist_ok=True)

data_dir = Path('/content/LaflaAI/data')
data_dir.mkdir(parents=True, exist_ok=True)
manifest_target = data_dir / 'veri_manifesti.json'
train_target = data_dir / 'train.jsonl'

def _find_candidates(patterns, roots):
    found = []
    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            for path in root.rglob(pattern):
                if path.is_file() and path not in found and '/.ipynb_checkpoints/' not in str(path):
                    found.append(path)
    return found

roots = [Path('/content'), Path('/content/gdrive/MyDrive')]
if not manifest_target.exists():
    manifest_candidates = [p for p in _find_candidates(['veri_manifesti.json', 'manifest.json', '*manifest*.json'], roots) if p != manifest_target]
    if len(manifest_candidates) == 1:
        shutil.copy2(manifest_candidates[0], manifest_target)
        print('Manifest kopyalandi:', manifest_candidates[0], '->', manifest_target)
    else:
        print('Manifest adaylari:', [str(p) for p in manifest_candidates[:20]])
        raise FileNotFoundError('Gercek veri manifesti bulunamadi veya birden fazla aday var. Dogru dosyayi /content/LaflaAI/data/veri_manifesti.json olarak koyun.')

if not train_target.exists():
    train_candidates = [p for p in _find_candidates(['train.jsonl', '*train*.jsonl', '*.jsonl'], roots) if p != train_target]
    train_candidates = sorted(train_candidates, key=lambda p: p.stat().st_size, reverse=True)
    if len(train_candidates) == 1:
        shutil.copy2(train_candidates[0], train_target)
        print('Train JSONL kopyalandi:', train_candidates[0], '->', train_target)
    else:
        print('Train JSONL adaylari:', [(str(p), p.stat().st_size) for p in train_candidates[:20]])
        raise FileNotFoundError('Gercek train.jsonl bulunamadi veya birden fazla aday var. Dogru dosyayi /content/LaflaAI/data/train.jsonl olarak koyun.')

print('Gercek veri hazir:', manifest_target, train_target)

In [ ]:
!PYTHONPATH=src python -m lafla_ai_core.cli.check_environment --optimizer adamw8bit
!PYTHONPATH=src python -m lafla_ai_core.cli.quality_scan --root .
!PYTHONPATH=src python -m lafla_ai_core.cli.preflight configs/model/lafla-400m-thinking.yaml configs/training/colab-t4-400m-4000.yaml configs/tokenizer/turkish-thinking-bpe.yaml configs/runtime/desktop-cpu-4bit.yaml configs/post_training/lafla-thinking-sft.yaml

In [ ]:
import yaml
from pathlib import Path

base_config = Path('configs/training/colab-t4-400m-4000.yaml')
step100_config = Path('/content/LaflaAI/reports/colab-t4-step100.yaml')
payload = yaml.safe_load(base_config.read_text(encoding='utf-8'))
payload['training'].update({
    'max_steps': 100,
    'warmup_steps': 10,
    'save_every': 100,
    'keep_last_checkpoints': 1,
    'log_every': 1,
})
step100_config.write_text(yaml.safe_dump(payload, sort_keys=False), encoding='utf-8')
print(step100_config.read_text(encoding='utf-8'))

In [ ]:
!PYTHONPATH=src python -m lafla_ai_core.cli.data_audit --manifest /content/LaflaAI/data/veri_manifesti.json --report /content/LaflaAI/reports/data-audit.json
!PYTHONPATH=src python -m lafla_ai_core.cli.tokenizer_train --config configs/tokenizer/turkish-thinking-bpe.yaml --output /content/LaflaAI/tokenizer/lafla-tokenizer.json --report /content/LaflaAI/reports/tokenizer-quality.json configs/data/lafla-model-identity.jsonl /content/LaflaAI/data/train.jsonl
!PYTHONPATH=src python -m lafla_ai_core.cli.hf_package --tokenizer-json /content/LaflaAI/tokenizer/lafla-tokenizer.json --output-dir /content/LaflaAI/hf-package --model-config configs/model/lafla-400m-thinking.yaml --model-name lafla-400m-thinking

In [ ]:
!PYTHONPATH=src python -m lafla_ai_core.cli.train_pretrain --model-config configs/model/lafla-400m-thinking.yaml --training-config /content/LaflaAI/reports/colab-t4-step100.yaml --tokenizer-path /content/LaflaAI/tokenizer/lafla-tokenizer.json --checkpoint-dir /content/LaflaAI/checkpoints --health-log /content/LaflaAI/reports/train-health-step100.jsonl --data-jsonl /content/LaflaAI/data/train.jsonl
!ls -lh /content/LaflaAI/checkpoints
!test -d /content/LaflaAI/checkpoints/lafla-step-000100

In [ ]:
!PYTHONPATH=src python -m lafla_ai_core.cli.test_checkpoint --checkpoint-dir /content/LaflaAI/checkpoints/lafla-step-000100 --tokenizer-path /content/LaflaAI/tokenizer/lafla-tokenizer.json --prompt "2+2 kac eder? Kisa cevap ver." --max-new-tokens 64 > /content/LaflaAI/reports/checkpoint-100-test.json
!cat /content/LaflaAI/reports/checkpoint-100-test.json

In [ ]:
!PYTHONPATH=src python -m lafla_ai_core.cli.artifact_manifest --root /content/LaflaAI --output /content/LaflaAI/reports/artifact-manifest-step100.json
!mkdir -p /content/gdrive/MyDrive/LaflaAI/final-checkpoint
!tar -czf /content/gdrive/MyDrive/LaflaAI/final-checkpoint/lafla-step100-tested.tar.gz -C /content/LaflaAI/checkpoints lafla-step-000100 lafla-final -C /content/LaflaAI tokenizer reports hf-package
!sync && ls -lh /content/gdrive/MyDrive/LaflaAI/final-checkpoint/lafla-step100-tested.tar.gz